# Phòng thực hành lượng tử nâng cao

18 ví dụ cho sáu học phần. Dự đoán trên giấy trước khi chạy từng ô. Python 3.12. Các ô độc lập, có thể chạy lại riêng.

Môi trường đã chạy: numpy 2.5.5, qiskit 2.6.2, pennylane 0.45.1. Trong Jupyter, nếu thiếu gói, chạy %pip install numpy qiskit pennylane trong một ô rồi khởi động lại kernel.

## Bài 01 · Đổi thước đo, dấu i đi đâu?

Đổi dấu ở hàng thứ hai rồi dự đoán cột xác suất nào sẽ hoán đổi trước khi chạy.

In [1]:
import numpy as np
B = np.array([[1, 1], [1j, -1j]]) / np.sqrt(2)
for psi in B.T:
    print("Z =", np.abs(psi)**2, "  Y =", np.round(np.abs(B.conj().T @ psi)**2, 8))
assert np.allclose(B.conj().T @ B, np.eye(2))

Z = [0.5 0.5]   Y = [1. 0.]
Z = [0.5 0.5]   Y = [0. 1.]


## Bài 02 · Máy có thể chỉ một số khác cả hai trị riêng không?

Một máy có thể chỉ mãi hai mức, dù trung bình là mức nằm giữa chúng. Hai cách đọc kết quả bên dưới tương ứng với hai đại lượng đó.

In [2]:
import numpy as np
A = np.array([[2, 1], [1, 2]])
values, vectors = np.linalg.eigh(A)
p = np.abs(vectors.conj().T @ [1, 0])**2
print("E/ε =", values, "  p =", p, "  trung bình =", values @ p)
assert np.allclose(p, [0.5, 0.5])

E/ε = [1. 3.]   p = [0.5 0.5]   trung bình = 1.9999999999999996


## Bài 03 · Vì sao mật độ có thể lớn hơn một?

Thử chia thành mười ô, giữ nguyên vùng vật lý từ 0,1 đến 0,3 µm. Khi đó phải cộng bốn ô nhỏ.

In [3]:
import numpy as np
edges = np.linspace(0, 0.5, 6)  # micromét
p = 2 * np.diff(edges)
print("Từng ô:", p, "  vùng từ 0,1 tới 0,3:", p[1:3].sum())
assert np.isclose(p.sum(), 1)

Từng ô: [0.2 0.2 0.2 0.2 0.2]   vùng từ 0,1 tới 0,3: 0.4000000000000001


## Bài 04 · Kéo giếng thế rộng gấp đôi có làm năng lượng giảm một nửa?

Hãy nhìn hiệu so với π² khi giảm bước lưới. Đây là sai số rời rạc hóa ngay cả khi phép giải trị riêng đã đủ chính xác.

In [4]:
import numpy as np
for M in (2, 8, 32, 96):
    side = -np.ones(M - 1)
    H = (M + 1)**2 * (2*np.eye(M) + np.diag(side, 1) + np.diag(side, -1))
    E1 = np.linalg.eigvalsh(H)[0]
    print(f"{M:2} điểm: E₁ = {E1:.6f}, sai số = {abs(E1-np.pi**2):.6f}")

2 điểm: E₁ = 9.000000, sai số = 0.869604
 8 điểm: E₁ = 9.769795, sai số = 0.099809
32 điểm: E₁ = 9.862153, sai số = 0.007452
96 điểm: E₁ = 9.868742, sai số = 0.000863


## Bài 05 · Hai mức năng lượng đứng yên, vì sao mật độ lại chuyển động?

Mình bỏ pha chung của mức thứ nhất khi tính. Tổng diện tích vẫn là một ở cả bốn thời điểm.

In [5]:
import numpy as np
x = np.linspace(0, 1, 1001)  # tính độ dài theo L
u, v = np.sqrt(2)*np.sin(np.pi*x), np.sqrt(2)*np.sin(2*np.pi*x)
for t in [0, 0.25, 0.5, 1]:  # thời gian theo T
    rho = np.abs((u + v*np.exp(-2j*np.pi*t))/np.sqrt(2))**2
    print(f"t/T = {t:g}, Lρ(L/4) = {rho[250]:.6f}")
    assert np.isclose(np.trapezoid(rho, x), 1)

t/T = 0, Lρ(L/4) = 2.914214
t/T = 0.25, Lρ(L/4) = 1.500000
t/T = 0.5, Lρ(L/4) = 0.085786
t/T = 1, Lρ(L/4) = 2.914214


## Bài 06 · Vì sao xoay máy đo 60° lại phải tính cos của 30°?

Các vectơ riêng là các cột của ma trận trả về. Hàng đầu cho biên độ chiếu của spin lên theo z lên từng vectơ ấy.

In [6]:
import numpy as np
a = np.deg2rad(60)
M = np.array([[np.cos(a), np.sin(a)], [np.sin(a), -np.cos(a)]])
e, v = np.linalg.eigh(M)
print("Kênh:", e, "  xác suất:", np.abs(v[0, :])**2)
assert np.isclose(abs(v[0, -1])**2, 0.75)

Kênh: [-1.  1.]   xác suất: [0.25 0.75]


## Bài 07 · Một hiệu năng lượng tạo ra cổng quay như thế nào?

Mình giữ đơn vị mili giây xuyên suốt vòng lặp. Nhầm sang giây sẽ làm mất ba bậc độ lớn.

In [7]:
import numpy as np
plus = np.ones(2)/np.sqrt(2)
for t_ms in [0, 0.25, 0.5, 1]:
    angle = 2*np.pi*t_ms  # tần số 1 kHz
    psi = plus * np.exp(-0.5j*angle*np.array([1, -1]))
    print(f"{t_ms:g} ms: P(+x)={abs(plus @ psi)**2:.3f}, P(0z)={abs(psi[0])**2:.3f}")

0 ms: P(+x)=1.000, P(0z)=0.500
0.25 ms: P(+x)=0.500, P(0z)=0.500
0.5 ms: P(+x)=0.000, P(0z)=0.500
1 ms: P(+x)=1.000, P(0z)=0.500


## Bài 08 · Bốn trăm phép đo có bắt buộc cho đúng ba trăm số 0?

Đổi seed để có một sổ đo khác. Giá trị lý tưởng phải giữ nguyên khi chỉ thay seed.

In [8]:
import numpy as np
import pennylane as qml
dev = qml.device("default.qubit", wires=1, seed=73)

@qml.qnode(dev)
def probabilities(angle):
    qml.RY(angle, wires=0)
    return qml.probs(wires=[0])

@qml.set_shots(shots=400)
@qml.qnode(dev)
def counts(angle):
    qml.RY(angle, wires=0)
    return qml.counts(wires=[0])

print("Lý tưởng:", probabilities(np.pi/3))
print("Một đợt lấy mẫu:", counts(np.pi/3))
assert np.allclose(probabilities(np.pi/3), [0.75, 0.25])

Lý tưởng: [0.75 0.25]
Một đợt lấy mẫu: {np.str_('0'): np.int64(302), np.str_('1'): np.int64(98)}


## Bài 09 · Theo dõi riêng một nửa cặp Bell thì mất thông tin gì?

Phép reshape tách chỉ số hàng thành hai nhãn và chỉ số cột thành hai nhãn. Phải lấy vết trên đúng hai trục của Bình.

In [9]:
import numpy as np
psi = np.array([1, 0, 0, 1])/np.sqrt(2)
bell = np.outer(psi, psi.conj())
mixed = np.diag([0.5, 0, 0, 0.5])
X = np.array([[0, 1], [1, 0]])
for rho in (bell, mixed):
    local = np.trace(rho.reshape(2, 2, 2, 2), axis1=1, axis2=3)
    print("ρ_A =", local.tolist(), "  ⟨XX⟩ =", round(np.trace(rho @ np.kron(X, X)), 4))
    assert np.allclose(local, np.eye(2)/2)

ρ_A = [[0.4999999999999999, 0.0], [0.0, 0.4999999999999999]]   ⟨XX⟩ = 1.0
ρ_A = [[0.5, 0.0], [0.0, 0.5]]   ⟨XX⟩ = 0.0


## Bài 10 · Hai phong bì có thể hẹn trước với nhau đến mức nào?

Vòng lặp duyệt đủ 16 chiến lược xác định. Bạn có thể tự gán trọng số cho chúng và thử đẩy trung bình vượt 2.

In [10]:
from collections import Counter
from itertools import product
scores = Counter(a*b + a*d + c*b - c*d for a,c,b,d in product((-1,1), repeat=4))
print(sorted(scores.items()))
assert scores == {-2: 8, 2: 8}

[(-2, 8), (2, 8)]


## Bài 11 · Bốn cặp hướng đo nào cho giá trị CHSH bằng 2√2?

Mỗi dòng ghi một cặp góc và bốn số đếm. Hãy lấy máy tính tay dựng lại riêng tương quan ở dòng cuối trước khi xem tổng.

In [11]:
import numpy as np
rng = np.random.default_rng(31)
A, B = [0, 90], [45, -45]
signs = np.array([1, 1, 1, -1])
ideal, sample = [], []
for a in A:
    for b in B:
        E = np.cos(np.deg2rad(a-b))
        # Cặp Bell có phân bố biên đều. Thứ tự kết quả: 00, 01, 10, 11.
        probabilities = (1 + E*np.array([1,-1,-1,1]))/4
        counts = rng.multinomial(400, probabilities)
        ideal.append(E)
        sample.append(counts @ [1,-1,-1,1]/400)
        print(a, b, counts.tolist())
print("S lý tưởng và S mẫu:", signs @ ideal, signs @ sample)
assert np.isclose(signs @ ideal, 2*np.sqrt(2))

0 45 [188, 29, 26, 157]
0 -45 [159, 30, 35, 176]
90 45 [182, 25, 26, 167]
90 -45 [19, 179, 171, 31]
S lý tưởng và S mẫu: 2.82842712474619 2.895


## Bài 12 · Làm thế nào biến một hàm nhiều-vào-một thành cổng lượng tử?

Số nguyên đánh chỉ mục cột đầu vào có dây đích ở bit trọng số 4. Dự đoán đầu ra cho cột số 6 trước khi chạy.

In [12]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
oracle = QuantumCircuit(3)
oracle.cx(0, 2)
oracle.cx(1, 2)
U = Operator(oracle).data
for x in range(4):
    for y in (0, 1):
        j = 4*y + x
        k = int(np.argmax(np.abs(U[:, j])))
        assert k == 4*(y ^ (x.bit_count() % 2)) + x
        print(f"x={x:02b}, y={y} → y′={k//4}")

x=00, y=0 → y′=0
x=00, y=1 → y′=1
x=01, y=0 → y′=1
x=01, y=1 → y′=0
x=10, y=0 → y′=1
x=10, y=1 → y′=0
x=11, y=0 → y′=0
x=11, y=1 → y′=1


## Bài 13 · Vì sao đổi bit phụ lại làm nhánh dữ liệu đổi dấu?

Nhân bảng hai thanh ghi với vectơ của dây phụ giúp đọc lại bốn biên độ dữ liệu kèm dấu.

In [13]:
import numpy as np
minus = np.array([1, -1])/np.sqrt(2)
joint = np.array([minus if x.bit_count()%2 == 0 else minus[::-1] for x in range(4)])/2
data = joint @ minus.conj()
print("Biên độ:", np.round(data, 3), "  xác suất:", np.round(data**2, 3))
assert np.allclose(joint, np.outer(data, minus))

Biên độ: [ 0.5 -0.5 -0.5  0.5]   xác suất: [0.25 0.25 0.25 0.25]


## Bài 14 · Một truy vấn đủ phân biệt hàm hằng với hàm cân bằng bằng cách nào?

Mạch chỉ đọc phân bố biên của hai dây đầu. So hai ô khác 0 ở hai lần chạy rồi tra nhãn nhị phân của chúng.

In [14]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
for parity in (False, True):
    qc = QuantumCircuit(3)
    qc.x(2)
    qc.h([0, 1, 2])
    if parity:  # hàm hằng 0 dùng oracle đồng nhất
        qc.cx(0, 2)
        qc.cx(1, 2)
    qc.h([0, 1])
    p = Statevector.from_instruction(qc).probabilities(qargs=[0, 1])
    print("Chẵn lẻ" if parity else "Hằng 0", np.round(p, 5))
    assert np.isclose(p[3 if parity else 0], 1)

Hằng 0 [1. 0. 0. 0.]
Chẵn lẻ [0. 0. 0. 1.]


## Bài 15 · Với 16 ngăn, nên dừng Grover ở vòng nào?

Mỗi vòng kiểm tra hai phép tính độc lập, một bên cập nhật cả 16 biên độ, một bên chỉ cộng góc.

In [15]:
import numpy as np
psi = np.ones(16)/4
w = int("1011", 2)
theta = np.arcsin(1/4)
for k in range(5):
    print(k, round(psi[w]**2, 6))
    assert np.isclose(psi[w]**2, np.sin((2*k+1)*theta)**2)
    psi[w] *= -1
    psi = 2*psi.mean() - psi

0 0.0625
1 0.472656
2 0.908447
3 0.961319
4 0.581704


## Bài 16 · Một nhịp 3/8 vòng hiện thành số 3 như thế nào?

Đổi dấu trong số mũ của tín hiệu rồi chạy lại. Với pha 3/8, đỉnh sẽ đổi từ nhãn 3 sang nhãn 5.

In [16]:
import numpy as np
Q = 8
for phase in (3/8, 1/3):
    signal = np.exp(2j*np.pi*np.arange(Q)*phase)
    # fft dùng dấu trừ, đúng với QFT ngược theo quy ước của bài.
    p = np.abs(np.fft.fft(signal)/Q)**2
    print(f"φ={phase:.4f}", np.round(p, 4))
    assert np.isclose(p.sum(), 1)

φ=0.3750 [0. 0. 0. 1. 0. 0. 0. 0.]
φ=0.3333 [0.0156 0.0316 0.1749 0.6878 0.0469 0.0186 0.0126 0.0119]


## Bài 17 · Vì sao bài tìm bậc của 2 theo mô-đun 21 sinh ra sáu pha?

Độ dài chu kỳ được quyết định bởi phép nhân. Bạn có thể đổi cơ số và xem các đỉnh di chuyển. Bảng đầu vào được dựng từ chính số dư.

In [17]:
import numpy as np
Q, N = 64, 21
joint = np.zeros((Q, 32))
for x in range(Q):
    joint[x, pow(2, x, N)] = 1/np.sqrt(Q)
spectrum = np.abs(np.fft.fft(joint, axis=0, norm="ortho"))**2
p = spectrum.sum(axis=1)
print("P(11), P(32), P(53):", np.round(p[[11,32,53]], 6))
assert np.isclose(p.sum(), 1)

P(11), P(32), P(53): [0.114196 0.166992 0.114196]


## Bài 18 · Máy trả về 11, làm sao lấy được hai ước của 21?

Dòng in đầu tiên có thể hiện 0 thay cho 0/1 vì Python rút gọn cách hiển thị. Thuật toán vẫn xét mẫu số 1.

In [18]:
from fractions import Fraction
from math import gcd
a, b = 11, 64
h0, h1, k0, k1 = 0, 1, 1, 0
while b:
    q, rem = divmod(a, b)
    a, b = b, rem
    h0, h1 = h1, q*h1+h0
    k0, k1 = k1, q*k1+k0
    print(Fraction(h1, k1), "  dư:", pow(2, k1, 21))
    if k1 < 21 and pow(2, k1, 21) == 1:
        r = k1
        break
z = pow(2, r//2, 21)
print("Ước:", gcd(z-1, 21), gcd(z+1, 21))
assert r == 6

0   dư: 2
1/5   dư: 11
1/6   dư: 1
Ước: 7 3
